# Explore a run

Pick any saved run by name. `load_predictor` builds a generic `pato.inference.Predictor` for it — every pipeline goes through the same code path, with the train→infer asymmetry hidden inside each LightningModule's `to_inference_model()`. The notebook then loads the dataset's `val` split (from `config`), runs inference, and shows source / ground-truth / prediction side by side.

Works for `unet` and `sam` (both the frozen-head and end-to-end `sam_finetune` regimes), and any future pipeline whose `LightningModule` implements `to_inference_model() → nn.Module`.

Runs live one-folder-each under `runs/` — Hydra's per-job dir, holding `config.yaml`, `checkpoints/`, `.hydra/`, `train.log`, `wandb/`. `list_runs(paths.runs)` returns them oldest-first, so `[-1]` is the most recent run.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

from config import paths
from pato.dataset import DatasetViewer
from pato.experiments import (
    list_runs,
    load_predictor,
    load_run,
    source_dataset_root,
)
from pato.visualize import load_image, show_side_by_side

## 1. Pick a run

In [ ]:
list_runs(paths.runs)

In [ ]:
RUN_NAME = list_runs(paths.runs)[-1]   # most recent run — ← or paste any name from the list above
run_path = paths.runs / RUN_NAME
run = load_run(run_path)

# Pull out the bits the notebook reuses below.
pipeline_name = run.config["pipeline"]["name"]
dataset_root = run.config["dataset"]["dataset_root"]

print(f"Run        : {run.name}")
print(f"Pipeline   : {pipeline_name}")
print(f"Best ckpt  : {run.best_checkpoint().name if run.best_checkpoint() else '(none)'}")
print(f"Dataset    : {dataset_root}")
print(f"Net        : {run.config['net']['_target_']}")
print(f"LR         : {run.config['lr']['learning_rate']}")
run.config

## 2. Load the predictor

One generic `Predictor` regardless of pipeline. Internally: dispatches on `config.pipeline` to load the right LightningModule, calls its `to_inference_model()`, reads tile size + overlap from the cache metadata (or run config fallback).

In [ ]:
predictor = load_predictor(run_path)
print(f"pipeline    : {pipeline_name}")
print(f"target_size : {predictor.target_size}")
print(f"overlap     : {predictor.overlap}")
print(f"device      : {predictor.device}")

## 3. Load the val split and predict on a few samples

`source_dataset_root(run)` traces the run back to its **full-image** normalized dataset (e.g. a SAM-feature cache → `nmsc-2x`). Override `val_root` below to evaluate on a different normalized dataset (e.g. `paths.data_processed / "nmsc-5x"`) — useful for sanity-checking generalization across resolutions.

In [ ]:
val_root = source_dataset_root(run)          # auto: trace back to source full-image dataset
# val_root = paths.data_processed / "nmsc-5x"  # ← uncomment + edit to override

val = DatasetViewer(root=val_root, split="val")
print(f"val source: {val_root}")
print(f"val split : {len(val)} samples (full images)")

In [ ]:
sample_indices = [0, len(val) // 2, len(val) - 1]
for idx in sample_indices:
    sample = val[idx]
    predicted = predictor.predict(sample)
    print(f"--- {val.sample_ids[idx]} — image {sample.image.shape[:2]} ---")
    fig = show_side_by_side(
        sample.image, sample.mask, predicted,
        titles=["source", "ground truth", f"{pipeline_name} prediction"],
        mask_zmax=11,
    )
    fig.show()

## 4. Predict on an arbitrary image path

Change `image_path` to any image file you want — no ground truth needed.

In [ ]:
image_path = paths.nmsc_5x / "Images" / "BCC_1.tif"   # ← change me

image = load_image(image_path)
predicted = predictor.predict(image_path)
print(f"image: {image.shape}, prediction: {predicted.shape} {predicted.dtype}")

show_side_by_side(
    image, predicted,
    titles=["source", f"{pipeline_name} prediction"],
    mask_zmax=11,
)